# Advanced RAG — Integrated ingestion review

This notebook imports functions from `ingestor.py`. The features are now implemented inside `Ingestor`; these cells demonstrate how to inspect them. **The notebook does not automatically ingest or delete anything.**


In [ ]:
from pathlib import Path
from ingestor import Ingestor, IngestorConfig, chunk_source, make_payload

config = IngestorConfig()
# ingestor = Ingestor(config)  # Requires GOOGLE_API_KEY; opens persistent Qdrant


## 8 — Structure-aware chunking

Splits Markdown-style headings before recursive token-aware splitting; ordinary PDF/DOCX text without heading markers uses the recursive splitter. Set `config.STRUCTURE_AWARE = True`.


In [ ]:
# ingestor = Ingestor(config)
# chunks = chunk_source(ingestor.files[0], config.DATA_DIR, ingestor.splitter, structure_aware=True)
# [(c.metadata, c.page_content[:120]) for c in chunks[:5]]


## 10 — Exact duplicate handling

Reports exact duplicate text across incoming chunks and existing Qdrant points. Does not remove the source references or automatically delete records.


In [ ]:
# ingestor.exact_duplicate_report(chunks)


## 11 — Near-duplicate detection

Uses RapidFuzz for bounded candidate reporting. Install with `%pip install rapidfuzz`. Not semantic equivalence and no automatic deletion.


In [ ]:
# ingestor.near_duplicate_report(chunks, threshold=90.0, max_existing=200)


## 18 — Embedding reuse

`ingest_one()` reuses matching point vectors or identical content vectors before requesting Gemini embeddings. It updates SQLite after Qdrant writes.


In [ ]:
# ingestor.ingest_one(Path("data/example.txt")) # Writes to Qdrant + SQLite; Gemini API charges may apply


## 19 — Validation and interrupted-run recovery checks

Compare the SQLite manifest against Qdrant, inspect changed/missing source files, and detect orphan points. `cleanup_orphans()` defaults to dry-run. SQLite/Qdrant are still not transactionally atomic.


In [ ]:
# report = ingestor.validate_ingestion()
# print(report)
# ingestor.cleanup_orphans()  # dry-run: returns orphan IDs
# ingestor.cleanup_orphans(dry_run=False)  # DELETES orphaned Qdrant points; use only after inspection


## Simple workflow

Run after initializing `ingestor`. Refresh the scan whenever files change; source removals are explicit.


In [ ]:
# ingestor.scan_changes()
# for path in ingestor.changed_files:
#     ingestor.ingest_one(path)
# for sid in ingestor.removed_ids:
#     ingestor.remove_source(sid)
# print(ingestor.validate_ingestion())
# ingestor.close()
